# Forest Inventory from ALS

This notebook walks through a complete aerial LiDAR forest inventory:

1. Load ALS data (or use AbovePy for KY From Above tiles)
2. CSF ground classification
3. Canopy height model (CHM) via ground surface interpolation
4. Individual tree detection (CHM-watershed)
5. Coverage statistics: gap fraction, canopy cover
6. Export results

This workflow is used for carbon stock estimation, timber cruising, and
wildlife habitat assessment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import AerialCloud

## Option A: AbovePy (Kentucky forests)

In [ ]:
# Uncomment to use real KY From Above data via AbovePy
# from abovepy import search, download
# import tempfile
# from pathlib import Path
# from occulus.io import read
#
# tiles = search('lidar', county='menifee', phase=3)  # Red River Gorge area
# path = download(tiles[0], dest=Path(tempfile.mkdtemp()))
# cloud = read(path, platform='aerial', subsample=0.3)

print('(AbovePy block commented out — using synthetic data below)')

## Option B: Synthetic Forest Cloud

In [ ]:
rng = np.random.default_rng(7)

def make_forest(rng, area=200, n_trees=60, n_pts=60_000):
    x = rng.uniform(0, area, n_pts)
    y = rng.uniform(0, area, n_pts)
    # Gentle terrain
    z = 2 * np.sin(x / 60) + 1.5 * np.cos(y / 50) + rng.normal(0, 0.05, n_pts)
    
    # Random tree crowns (Gaussian height profile)
    tree_centers = rng.uniform(5, area - 5, (n_trees, 2))
    tree_heights = rng.uniform(8, 35, n_trees)  # 8–35 m trees
    crown_radii = tree_heights * 0.15  # crown radius proportional to height
    
    for (tx, ty), th, cr in zip(tree_centers, tree_heights, crown_radii):
        d = np.hypot(x - tx, y - ty)
        in_crown = d < cr * 1.5
        canopy_z = th * np.exp(-d[in_crown]**2 / (2 * cr**2))
        z[in_crown] = np.maximum(z[in_crown], z[in_crown] + canopy_z)
    
    return np.column_stack([x, y, z]).astype(np.float64), tree_centers, tree_heights

xyz, true_centers, true_heights = make_forest(rng)
cloud = AerialCloud(xyz)
print(f'Synthetic forest: {cloud.n_points:,} points')
print(f'True tree count : {len(true_centers)}')
print(f'Height range    : {true_heights.min():.1f}–{true_heights.max():.1f} m')

## Ground Classification

In [ ]:
from occulus.segmentation import classify_ground_csf

classified = classify_ground_csf(cloud)
if classified.classification is not None:
    n_g = (classified.classification == 2).sum()
    print(f'Ground: {n_g:,} ({n_g/cloud.n_points:.1%})')

## Canopy Height Model

In [ ]:
from occulus.metrics import canopy_height_model, coverage_statistics

chm, xe, ye = canopy_height_model(classified, resolution=0.5)
cov = coverage_statistics(classified, resolution=0.5)

valid = chm[~np.isnan(chm)]
print(f'CHM: {chm.shape[0]}×{chm.shape[1]} cells at 0.5 m')
print(f'Max canopy height : {valid.max():.1f} m')
print(f'Mean canopy height: {valid[valid > 2].mean():.1f} m (cells > 2 m)')
print(f'Gap fraction      : {cov.gap_fraction:.2%}')
print(f'Canopy cover      : {1 - cov.gap_fraction:.2%}')
print(f'Mean density      : {cov.mean_density:.1f} pts/m²')

## Individual Tree Segmentation

In [ ]:
from occulus.segmentation import segment_trees

seg = segment_trees(classified, resolution=0.5, min_height=3.0, min_crown_area=5.0)
print(f'Detected trees: {seg.n_segments}  (true: {len(true_centers)})')
print(f'Detection rate: {min(seg.n_segments, len(true_centers)) / len(true_centers):.1%} (approx.)')

# Cluster sizes
sizes = [(seg.labels == i).sum() for i in range(seg.n_segments)]
print(f'Points per tree: {np.mean(sizes):.0f} avg, {min(sizes)}–{max(sizes)} range')

## Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# CHM
im0 = axes[0].imshow(chm, origin='lower', cmap='Greens',
                     extent=[xe[0], xe[-1], ye[0], ye[-1]])
plt.colorbar(im0, ax=axes[0], label='Height (m)')
axes[0].set_title('Canopy Height Model (0.5 m)'); axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')

# Segmented trees (top view, coloured by label)
labels = seg.labels
cmap_pts = plt.cm.tab20
above = labels != -1
axes[1].scatter(xyz[above, 0], xyz[above, 1], 
                c=labels[above] % 20, cmap=cmap_pts, s=0.5, alpha=0.6)
axes[1].scatter(true_centers[:, 0], true_centers[:, 1], 
                marker='+', color='black', s=30, zorder=5, label='True centers')
axes[1].set_title(f'Tree Segmentation ({seg.n_segments} trees)')
axes[1].set_xlabel('X (m)'); axes[1].legend(markerscale=1)

# Height histogram
axes[2].hist(true_heights, bins=15, color='forestgreen', edgecolor='white', alpha=0.8, label='True')
axes[2].set_title('Tree Height Distribution')
axes[2].set_xlabel('Height (m)'); axes[2].set_ylabel('Count')
axes[2].legend()

plt.suptitle('Forest Inventory — CHM + Individual Tree Detection', fontsize=13)
plt.tight_layout()
plt.savefig('../../outputs/forest_inventory.png', dpi=150)
plt.show()

## Summary Table

In [ ]:
from occulus.metrics import compute_cloud_statistics
stats = compute_cloud_statistics(cloud)

print('\n=== Forest Inventory Summary ===')
print(f'  Total points      : {cloud.n_points:,}')
print(f'  Z range           : {stats.z_min:.1f} – {stats.z_max:.1f} m')
print(f'  Trees detected    : {seg.n_segments}')
print(f'  Max tree height   : {valid.max():.1f} m')
print(f'  Canopy cover      : {1-cov.gap_fraction:.1%}')
print(f'  Mean point density: {cov.mean_density:.1f} pts/m²')

**Next**: `06_urban_analysis.ipynb`